In [0]:
%sql
CREATE TABLE IF NOT EXISTS solutions_catalog.pih_schema.brand_data_embeddings (
    id BIGINT GENERATED ALWAYS AS IDENTITY,
    study_id BIGINT,
    brand_name STRING,
    advertiser_name STRING,
    kpi_metric_name STRING,
    metric_display_name STRING,
    mapping STRING,
    platform STRING,
    control_pct DOUBLE,
    exposed_pct DOUBLE,
    lift_pct DOUBLE,
    update_timestamp TIMESTAMP,
    embeddings ARRAY<FLOAT>
)
USING DELTA;


In [0]:
%pip install mlflow[databricks] --upgrade
dbutils.library.restartPython()


In [0]:
from pyspark.sql.functions import pandas_udf, col, max as spark_max,lit
from pyspark.sql.types import ArrayType, FloatType
import pandas as pd
from mlflow.deployments import get_deploy_client
from pyspark.sql.functions import concat_ws, col

# Table names
source_table = "solutions_catalog.pih_schema.brand_data"
target_table = "solutions_catalog.pih_schema.brand_data_embeddings"

# Step 1: Load max update timestamp from embeddings table (if exists)
try:
    max_timestamp = spark.table(target_table).agg(spark_max("update_timestamp")).collect()[0][0]
except:
    max_timestamp = None

# Step 2: Load only new or changed rows from source
df = spark.table(source_table).select(
    "study_id", "brand_name", "advertiser_name", "kpi_metric_name",
    "metric_display_name", "mapping", "platform",
    "control_pct", "exposed_pct", "lift_pct", "update_timestamp"
)

if max_timestamp:
    df = df.filter(col("update_timestamp") > max_timestamp)

if df.isEmpty():
    print("✅ No new rows to embed.")
else:
    # Step 3: Create input text column
    # Add a rich "embedding_input" column with structured context
    df = df.withColumn(
        "embedding_input",
        concat_ws(
            " ",
            concat_ws(": ", lit("Study ID"), col("study_id").cast("string")),
            concat_ws(": ", lit("Brand"), col("brand_name")),
            concat_ws(": ", lit("Advertiser"), col("advertiser_name")),
            concat_ws(": ", lit("KPI"), col("kpi_metric_name")),
            concat_ws(": ", lit("Metric"), col("metric_display_name")),
            concat_ws(": ", lit("Mapping"), col("mapping")),
            concat_ws(": ", lit("Platform"), col("platform")),
            concat_ws(": ", lit("Control"), col("control_pct").cast("string")),
            concat_ws(": ", lit("Exposed"), col("exposed_pct").cast("string")),
            concat_ws(": ", lit("Lift"), col("lift_pct").cast("string"))
        )
    )

    # Step 4: Define embedding function
    deploy_client = get_deploy_client("databricks")

    @pandas_udf(ArrayType(FloatType()))
    def get_embeddings(texts: pd.Series) -> pd.Series:
        from mlflow.deployments import get_deploy_client

        # Clean up input
        clean_texts = texts.fillna("").astype(str).replace("nan", "", regex=False)
        clean_texts = clean_texts.apply(lambda x: x.strip() if x.strip() != "" else "[EMPTY TEXT]")

        deploy_client = get_deploy_client("databricks")
        batch_size = 150
        inputs = clean_texts.tolist()
        all_embeddings = []

        for i in range(0, len(inputs), batch_size):
            chunk = inputs[i:i + batch_size]
            result = deploy_client.predict(endpoint="databricks-gte-large-en", inputs={"input": chunk})

            if isinstance(result["data"], list):
                # Case 1: list of vectors
                if all(isinstance(item, list) for item in result["data"]):
                    all_embeddings.extend(result["data"])
                # Case 2: list of dicts with 'embedding' key
                elif all(isinstance(item, dict) and "embedding" in item for item in result["data"]):
                    all_embeddings.extend([item["embedding"] for item in result["data"]])
                else:
                    raise ValueError("Unexpected format in 'data' field of response.")
            else:
                raise ValueError("Expected 'data' to be a list.")

        return pd.Series(all_embeddings)




    # Step 5: Generate and store embeddings
    df_with_embeddings = df.withColumn("embeddings", get_embeddings("embedding_input")).drop("embedding_input")
    
    from pyspark.sql.functions import col

    # Ensure update_timestamp column is of the correct type
    df_with_embeddings = df_with_embeddings.withColumn("update_timestamp", col("update_timestamp").cast("timestamp"))

    # Then append to the Delta table
    df_with_embeddings.write.mode("overwrite").format("delta").saveAsTable(target_table)

    print(f"✅ Appended {df_with_embeddings.count()} new rows with embeddings to {target_table}")


In [0]:
%sql

select * from solutions_catalog.pih_schema.brand_data_embeddings


In [0]:
%sql

select distinct platform from solutions_catalog.pih_schema.brand_data_embeddings